# Weather-forecasting demo

Inspect several held-out, chronological forecasts from every RNN model and the persistence baseline.

## Run training once (only if needed)

Run the next cell only if saved predictions are absent or you need a fresh experiment. If it has already run, skip it because this walkthrough loads `outputs/predictions.npz`.

In [ ]:
# Run only when saved artifacts are missing or need regeneration.
!cd .. && python scripts/train.py && python scripts/build_report_assets.py

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
OUTPUTS = Path('../outputs')
if not (OUTPUTS / 'predictions.npz').exists():
    raise FileNotFoundError('Run the optional training cell first; outputs/predictions.npz is missing.')
data = np.load(OUTPUTS / 'predictions.npz')
models = ['lstm', 'gru', 'seq2seq_lstm', 'attention_seq2seq', 'persistence']
pd.DataFrame({'test_forecast_start': pd.to_datetime(data['dates'][:5])})

In [ ]:
examples = np.linspace(0, len(data['targets']) - 1, 3, dtype=int)
colors = {'lstm':'#4C78A8', 'gru':'#59A14F', 'seq2seq_lstm':'#F28E2B', 'attention_seq2seq':'#E45756', 'persistence':'#9D755D'}
names = {'lstm':'LSTM', 'gru':'GRU', 'seq2seq_lstm':'Seq2Seq', 'attention_seq2seq':'Attention Seq2Seq', 'persistence':'Persistence'}
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for column, index in enumerate(examples):
    for row, target in enumerate(['Temperature (°C)', 'Humidity (%)']):
        axis = axes[row, column]
        axis.plot(range(1, 73), data['targets'][index, :, row], color='black', linewidth=2, label='Observed')
        for model in models:
            axis.plot(range(1, 73), data[model][index, :, row], color=colors[model], linewidth=1, label=names[model])
        axis.set_title(str(pd.to_datetime(data['dates'][index]))[:10]); axis.set_ylabel(target); axis.grid(alpha=.25)
        if row == 1: axis.set_xlabel('Hours ahead')
handles, labels = axes[0,0].get_legend_handles_labels(); fig.legend(handles, labels, loc='lower center', ncol=3); fig.tight_layout(rect=(0,.08,1,1))

In [ ]:
index = examples[-1]
horizon = 72
rows = []
for model in models:
    for target_index, target in enumerate(['T (degC)', 'rh (%)']):
        error = abs(data[model][index, horizon-1, target_index] - data['targets'][index, horizon-1, target_index])
        rows.append({'model': names[model], 'target': target, '72-hour absolute error': error})
pd.DataFrame(rows).round(3)